In [1]:
"""
By: John Callinan and Tajvir Dhillon

Sim of an 8-hour ER shift. Patients generated at poisson-distributed intervals 
with severities 1 to 5 with 1 being the worst and 5 being the least severe. 
Treated in priority order: more severe first ties are broken by
earliest arrival. Results saved to CSV and summary metrics printed.

Jack Remember this is how to run it and run the tests:
python er_simulation.py      
python -m pytest er_simulation.py -v   
"""

'\nBy: John Callinan and Tajvir Dhillon\n\nSim of an 8-hour ER shift. Patients generated at poisson-distributed intervals \nwith severities 1 to 5 with 1 being the worst and 5 being the least severe. \nTreated in priority order: more severe first ties are broken by\nearliest arrival. Results saved to CSV and summary metrics printed.\n\nJack Remember this is how to run it and run the tests:\npython er_simulation.py      \npython -m pytest er_simulation.py -v   \n'

In [2]:
import csv
import heapq
import os
import random
import time
import numpy as np
import pandas as pd
from patients import Patient
from er_queue import ERQueue, QueueError
from sim import generate_patients, run_simulation
from csv_l_w import save_patients_to_csv, load_patients_from_csv
from anaylsis import compute_metrics, analyze_results

In [3]:
#Just a banner
print("=" * 60)
print("EMERGENCY ROOM QUEUE SIMULATION")
print("=" * 60)

EMERGENCY ROOM QUEUE SIMULATION


In [4]:
#figure out where all the files are and build file path
here = os.getcwd()
roster_csv = os.path.join(here, "patients_roster.csv")
results_csv = os.path.join(here, "patients_results.csv")

# 1. Generate patient roster
print("\n[1] Generating patient roster (8-hour shift, ~6 arrivals/hr)...")
roster = list(generate_patients(shift_minutes=480, avg_arrivals_per_hour=6.0, seed=42))
save_patients_to_csv(roster, roster_csv) #build roseter than save it 
print(f"    Generated {len(roster)} patients -> {roster_csv}")


[1] Generating patient roster (8-hour shift, ~6 arrivals/hr)...
    Generated 50 patients -> C:\Users\jack\zfolder\patients_roster.csv


In [5]:
# 2. Reload roster from CSV
print("\n[2] Reloading roster from CSV...")
patients = load_patients_from_csv(roster_csv)
print(f"    Loaded {len(patients)} patients.")


[2] Reloading roster from CSV...
    Loaded 50 patients.


In [6]:
# 3. Run simulation 
print("\n[3] Running simulation with 2 doctors...")
start = time.perf_counter() #start timer
results = run_simulation(patients, num_doctors=2, seed=7) #run sim
elapsed = time.perf_counter() - start #figuer out time elapsed
print(f"    Treated {len(results)} patients in {elapsed*1000:.2f} ms wall time.") #print time elapsed


[3] Running simulation with 2 doctors...
    Treated 50 patients in 1.59 ms wall time.


In [7]:
# 4. Save results to CSV
save_patients_to_csv(results, results_csv)
print(f"    Results saved -> {results_csv}")

    Results saved -> C:\Users\jack\zfolder\patients_results.csv


In [8]:
# 5. Print the metrics formated neatly 
print("\n[4] Summary metrics:")
for key, value in compute_metrics(results).items():
    print(f"    {key:30s} = {value:7.2f}")


[4] Summary metrics:
    total_patients                 =   50.00
    avg_wait_minutes               =   10.97
    median_wait_minutes            =    4.21
    max_wait_minutes               =   73.43
    min_wait_minutes               =    0.00
    stdev_wait_minutes             =   16.74
    avg_wait_sev1                  =    3.56
    avg_wait_sev2                  =    6.82
    avg_wait_sev3                  =    4.51
    avg_wait_sev4                  =   10.58
    avg_wait_sev5                  =   21.02


In [9]:
# 6. Print the post sim analysis (filter/lambda/map/list comp)
print("\n[5] Post-simulation analysis:")
for key, value in analyze_results(results).items():
    if isinstance(value, float):
        print(f"    {key:35s} = {value:7.2f}")
    else:
        print(f"    {key:35s} = {value}")


[5] Post-simulation analysis:
    critical_patient_count              = 7
    critical_avg_wait                   =    5.89
    low_acuity_avg_wait                 =   14.62
    projected_avg_wait_with_shortage    =   13.17


In [10]:
# 7. Shows a sample of critical patients 
print("\n[6] Sample patient log (first 5 critical cases):")
for p in [p for p in results if p.severity <= 2][:5]:
    print(f"    {p}")

print("\nDone.")


[6] Sample patient log (first 5 critical cases):
    Patient #0008 | sev=2 | arrived= 94.90m | wait=0.0m | treat=37.78m
    Patient #0013 | sev=1 | arrived=145.85m | wait=5.0m | treat=54.62m
    Patient #0025 | sev=1 | arrived=228.95m | wait=2.1m | treat=37.77m
    Patient #0024 | sev=2 | arrived=225.41m | wait=16.7m | treat=32.31m
    Patient #0029 | sev=2 | arrived=277.04m | wait=5.0m | treat=42.88m

Done.


In [11]:
!python -m pytest tests.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.3, pytest-7.4.4, pluggy-1.0.0 -- C:\Users\jack\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\jack\zfolder
plugins: anyio-4.2.0
collecting ... collected 10 items

tests.py::test_patient_invalid_severity PASSED                           [ 10%]
tests.py::test_patient_invalid_arrival_time PASSED                       [ 20%]
tests.py::test_patient_lt_orders_by_severity PASSED                      [ 30%]
tests.py::test_patient_lt_breaks_ties_by_arrival PASSED                  [ 40%]
tests.py::test_queue_pops_most_severe_first PASSED                       [ 50%]
tests.py::test_queue_same_severity_fifo PASSED                           [ 60%]
tests.py::test_queue_empty_raises PASSED                                 [ 70%]
tests.py::test_generator_invalid_shift PASSED                            [ 80%]
tests.py::test_wait_times_are_nonnegative_and_finite PASSED            